Pydantic AI is modular. If there is a tool that already exists in the Langchain ecosystem, you can easily wrap it and use it with your Pydantic AI agent.

To demonstrate, let's use the real LangChain Wikipedia tool!

# Install the necessary libraries for this demo
!pip install -q langchain-community wikipedia

In [1]:
# Install the necessary libraries for this demo
!pip install -q langchain-community wikipedia

Initializing the LangChain Tool
We pull the tool from langchain_community exactly as their docs describe.

In [2]:
from langchain_community.tools.wikipedia.tool import WikipediaQueryRun
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper

# Langchain tool setup
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
langchain_wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

Adapting it to Pydantic AI
We simply write a @agent.tool_plain wrapper that invokes the Langchain .invoke() method inside!

In [10]:
import nest_asyncio
nest_asyncio.apply()

from pydantic_ai import Agent
from dotenv import load_dotenv
load_dotenv()
research_agent = Agent(
    "groq:llama-3.1-8b-instant",
    system_prompt=(
        "You are a precise researcher. "
        "Use Wikipedia tool only when needed, then answer clearly."
    )
)


@research_agent.tool_plain
def search_wikipedia(search_string: str) -> str:
    print(f"\n[Wikipedia Tool] Query: {search_string}")
    return langchain_wiki_tool.invoke({"query": search_string})


print("--- Asking the Agent a History Question ---")

response = research_agent.run_sync(
    "What year was the James Webb Space Telescope launched? Summarize briefly."
)

print("\n--- Final Answer ---")
print(response.output)

--- Asking the Agent a History Question ---

[Wikipedia Tool] Query: James Webb Space Telescope launch date

--- Final Answer ---
The James Webb Space Telescope was launched on December 25, 2021.
